The code in this notebook creates a set of xyz files with model polynuclear ThxOy clusters from a bigger single cluster which is stored in 'Th40.xyz' file. In this paper, Th40 cluster is constructed from the Ce40 crystal structure (CSD refcode KENGUI) by extracting a single molecular cluster from the crytsl structure, stripping all the organic fragments and substituting Ce atoms with Th atoms.

The decision on how to slice an initial cluster is made based on the structure catalogue file {structure_catalogue.txt} created by structure_catalogue_maker() function from this paper {https://doi.org/10.1038/s41524-022-00896-3}. 

As the atoms to be dropped from the initial big structure when constructing each smaller structure are selected randomly, some of the resulting structures may not be connected and consist of several atom 'islands'. In that case, the island containing the largest number of heavy atoms is retained with all coordinated oxygen atoms, while the smaller islands are dropped.

The names of the prepared files are formatted so that they contain a number of Th atoms in the cluster as the first symbol for easier labelling of data before the network training.

In [14]:
import numpy as np
import os
import random

# Import path configuration
import sys
sys.path.insert(0, '..')  # Add parent directory to path
from config import get_path

In [15]:
def format_XYZ(xyz_file, allowed_atoms):
    """
    Read an xyz file and count the number of atoms of specified types.
    
    Parameters
    ----------
    xyz_file : str
        Path to the xyz file
    allowed_atoms : list
        List of atom types to count (e.g., ["Th"])
    
    Returns
    -------
    int
        Number of atoms matching the allowed types
    """
    atom_count = 0
    with open(xyz_file, 'r') as f:
        lines = f.readlines()
    
    for line in lines[2:]:  # Skip first two lines (atom count and comment)
        fields = line.split()
        if fields and fields[0] in allowed_atoms:
            atom_count += 1
    
    print(f"Found {atom_count} {allowed_atoms} atoms in {xyz_file}")
    return atom_count

In [ ]:
def structure_catalogue_maker(Number_of_structures, Number_of_atoms, lower_atom_number, higher_atom_number):
    """This function makes a shuffled list containing 'Number_of_structures' number of lists which each is 
    'Number_of_atoms' long and is randomly distributed with 0's and 1's whereas the minimum number of 1's are 
    'lower_atom_number' and the maximum number of 1's are 'higher_atom_number'."""
    
    print(
        "Starting to make a structure catalogue with: ",
        f"{str(Number_of_structures)} structure from the starting model.",
    )
    print(
        f"The structure will have between {str(lower_atom_number)} and {str(higher_atom_number)} atoms"
    )
    structure_catalogue = []
    for _ in range(Number_of_structures):
        one_count = random.randint(lower_atom_number, higher_atom_number)
        zero_count = Number_of_atoms  - one_count
        my_list = [0]*zero_count + [1]*one_count
        random.shuffle(my_list)
        my_list.insert(0, one_count)
        structure_catalogue.append(my_list)
    print ("Permutations Succeeded")
    return structure_catalogue
    

In [17]:
def make_xyz_catalogue(catalogue, initial_xyz, output_dir, metal_atom, threshold_MO, threshold_MM):
    """
    Create xyz files for polynuclear clusters from a parent cluster structure.
    
    Parameters
    ----------
    catalogue : list
        List of structure masks from structure_catalogue_maker (each is a list starting with atom count, then 0/1 mask)
    initial_xyz : str
        Path to the initial large cluster xyz file (e.g., th40.xyz or ce40.xyz)
    output_dir : Path
        Directory where output xyz files will be saved
    metal_atom : str
        Symbol of the metal atom (e.g., "Th" or "Ce")
    threshold_MO : float
        Maximum metal-O distance to consider oxygen coordinated to metal
    threshold_MM : float
        Maximum metal-metal distance to consider atoms in same cluster
    """
    with open(initial_xyz, 'r') as f:
        lines = f.readlines()

    # Get the number of atoms from the xyz file
    num_atoms = int(lines[0])
    
    created_count = 0

    # Iterate over the catalogue entries
    for k, entry in enumerate(catalogue):
        # entry[0] is the atom count, entry[1:] is the mask
        array = np.array(entry[1:])
        
        # Create a list to store the modified xyz lines
        new_lines = [f"{num_atoms}\n", lines[1]]

        # Iterate over the atoms in the xyz file
        for i, line in enumerate(lines[2:]):
            # If the index is less than the length of the array, process the atom according to the array value
            if i < len(array) and array[i] == 1 or i >= len(array):
                new_lines.append(line)

        metal_coordinates = []
        for line in new_lines[2:]:
            fields = line.split()
            if fields and fields[0] == metal_atom:
                metal_coordinates.append([float(fields[1]), float(fields[2]), float(fields[3])])

        metal_groups = []
        while metal_coordinates:
            metal_group = [metal_coordinates.pop()]
            for metal_coordinate in metal_coordinates:
                distance = np.linalg.norm(np.array(metal_group[0]) - np.array(metal_coordinate))
                if distance < threshold_MM:
                    metal_group.append(metal_coordinate)
            metal_groups.append(metal_group)

        final_result = []
        for element in metal_groups:
            found = False
            for result in final_result:
                if set(map(tuple, element)) & set(map(tuple, result)):
                    result.extend([e for e in element if e not in result])
                    found = True
                    break
            if not found:
                final_result.append(element)

        if metal_groups:
            largest_metal_group = max(final_result, key=lambda x: len(x))
            
            # Count metal atoms for filename
            metal_count = len(largest_metal_group)
            
            # Save to output directory with nuclearity as first character
            output_file = output_dir / f'{metal_count}_{k}.xyz'
            with open(output_file, 'w') as new_f:
                # Write correct atom count
                total_atoms = len(largest_metal_group)
                # Count O atoms that will be included
                for line in lines[2:]:
                    fields = line.split()
                    if fields and fields[0] == "O":
                        O_coordinate = [float(fields[1]), float(fields[2]), float(fields[3])]
                        for metal_coordinate in largest_metal_group:
                            distance = np.linalg.norm(np.array(metal_coordinate) - np.array(O_coordinate))
                            if distance < threshold_MO:
                                total_atoms += 1
                                break
                
                new_f.write(f"{total_atoms}\n")
                new_f.write(lines[1])  # Comment line
                for metal_coordinate in largest_metal_group:
                    new_f.write(f"{metal_atom} " + " ".join([str(x) for x in metal_coordinate]) + "\n")
                for line in lines[2:]:
                    fields = line.split()
                    if fields and fields[0] == "O":
                        O_coordinate = [float(fields[1]), float(fields[2]), float(fields[3])]
                        for metal_coordinate in largest_metal_group:
                            distance = np.linalg.norm(np.array(metal_coordinate) - np.array(O_coordinate))
                            if distance < threshold_MO:
                                new_f.write(line)
                                break
            created_count += 1
    
    print(f"Created {created_count} xyz files in {output_dir}")

In [20]:
# Set working directory to Th clusters (parent folder with input files)
th_clusters_path = get_path('th_clusters')
output_path = get_path('th_model_clusters')

# Ensure output directory exists
output_path.mkdir(parents=True, exist_ok=True)

os.chdir(th_clusters_path)
print(f"Working directory: {th_clusters_path}")
print(f"Output directory: {output_path}")
print(f"\nInput files expected:")
print(f"  - th40.xyz: {'exists' if (th_clusters_path / 'th40.xyz').exists() else 'missing'}")
print(f"  - structure_catalogue.txt: {'exists' if (th_clusters_path / 'structure_catalogue.txt').exists() else 'missing'}")

Working directory: /Users/dimitrygrebenyuk/Documents/GitHub/PDF-NN/complete workflow/pdf-nn-data/th_clusters
Output directory: /Users/dimitrygrebenyuk/Documents/GitHub/PDF-NN/complete workflow/pdf-nn-data/th_clusters/model_clusters

Input files expected:
  - th40.xyz: exists
  - structure_catalogue.txt: missing


In [6]:
starting_model = 'th40.xyz' # Name of the starting model file
Number_of_structures = 10000 # Number of structures made to the structure catalogue
allowed_atoms = ["Th"] # The atoms that should be permuted, can be multiple atoms
threshold = 3.0 # Threshold for W - O bond
print(starting_model)

th40.xyz


In [7]:
NumW = format_XYZ(starting_model, allowed_atoms) # Format the starting model and calculate then number of atoms that should be permuted in the starting model
structure_catalogue = structure_catalogue_maker(Number_of_structures, Number_of_atoms=NumW, lower_atom_number=2, higher_atom_number=9)
#print ("We show the first 10 structures in the catalogue:")
#structure_catalogue[:100]

Found 40 ['Th'] atoms in th40.xyz
Starting to make a structure catalogue with:  10000 structure from the starting model.
The structure will have between 2 and 9 atoms
Permutations Succeeded


In [ ]:
# Generate Th cluster xyz files from the catalogue
make_xyz_catalogue(
    catalogue=structure_catalogue, 
    initial_xyz='th40.xyz',
    output_dir=output_path,
    metal_atom="Th",
    threshold_MO=3.0, 
    threshold_MM=5.2
)

Created 10000 xyz files in /Users/dimitrygrebenyuk/Documents/GitHub/PDF-NN/complete workflow/pdf-nn-data/th_clusters/model_clusters


In [9]:
# Verify the created files
directory = get_path('th_model_clusters')
files = [f for f in os.listdir(directory) if f.endswith('.xyz')]
print(f"Total xyz files created: {len(files)}")

# Count by nuclearity
from collections import Counter
nuclearities = [int(f.split('_')[0]) for f in files]
counts = Counter(nuclearities)
print("\nDistribution by nuclearity:")
for nuc in sorted(counts.keys()):
    print(f"  {nuc} Th atoms: {counts[nuc]} structures")

Total xyz files created: 10000

Distribution by nuclearity:
  1 Th atoms: 2153 structures
  2 Th atoms: 2748 structures
  3 Th atoms: 2055 structures
  4 Th atoms: 1461 structures
  5 Th atoms: 832 structures
  6 Th atoms: 415 structures
  7 Th atoms: 230 structures
  8 Th atoms: 79 structures
  9 Th atoms: 27 structures


The same process is applied to create CexOy clusters from the Ce40 parent structure.

In [21]:
# Set working directory to Ce clusters
ce_clusters_path = get_path('ce_clusters')
ce_output_path = get_path('ce_model_clusters')

# Ensure output directory exists
ce_output_path.mkdir(parents=True, exist_ok=True)

os.chdir(ce_clusters_path)
print(f"Working directory: {ce_clusters_path}")
print(f"Output directory: {ce_output_path}")
print(f"\nInput files expected:")
print(f"  - ce40.xyz: {'exists' if (ce_clusters_path / 'ce40.xyz').exists() else 'missing'}")

Working directory: /Users/dimitrygrebenyuk/Documents/GitHub/PDF-NN/complete workflow/pdf-nn-data/ce_clusters
Output directory: /Users/dimitrygrebenyuk/Documents/GitHub/PDF-NN/complete workflow/pdf-nn-data/ce_clusters/model_clusters

Input files expected:
  - ce40.xyz: exists


In [11]:
# Parameters for Ce cluster generation
ce_starting_model = 'ce40.xyz'
ce_allowed_atoms = ["Ce"]
print(f"Starting model: {ce_starting_model}")

Starting model: ce40.xyz


In [12]:
# Generate structure catalogue for Ce clusters
NumCe = format_XYZ(ce_starting_model, ce_allowed_atoms)
ce_structure_catalogue = structure_catalogue_maker(Number_of_structures, Number_of_atoms=NumCe, lower_atom_number=2, higher_atom_number=9)

Found 40 ['Ce'] atoms in ce40.xyz
Starting to make a structure catalogue with:  10000 structure from the starting model.
The structure will have between 2 and 9 atoms
Permutations Succeeded


In [18]:
# Generate Ce cluster xyz files from the catalogue
make_xyz_catalogue(
    catalogue=ce_structure_catalogue, 
    initial_xyz='ce40.xyz',
    output_dir=ce_output_path,
    metal_atom="Ce",
    threshold_MO=3.0, 
    threshold_MM=5.2
)

Created 10000 xyz files in /Users/dimitrygrebenyuk/Documents/GitHub/PDF-NN/complete workflow/pdf-nn-data/ce_clusters/model_clusters


In [19]:
# Verify the created Ce cluster files
ce_directory = get_path('ce_model_clusters')
ce_files = [f for f in os.listdir(ce_directory) if f.endswith('.xyz')]
print(f"Total Ce xyz files created: {len(ce_files)}")

# Count by nuclearity
from collections import Counter
ce_nuclearities = [int(f.split('_')[0]) for f in ce_files]
ce_counts = Counter(ce_nuclearities)
print("\nDistribution by nuclearity:")
for nuc in sorted(ce_counts.keys()):
    print(f"  {nuc} Ce atoms: {ce_counts[nuc]} structures")

Total Ce xyz files created: 10000

Distribution by nuclearity:
  1 Ce atoms: 2186 structures
  2 Ce atoms: 2712 structures
  3 Ce atoms: 2063 structures
  4 Ce atoms: 1425 structures
  5 Ce atoms: 809 structures
  6 Ce atoms: 472 structures
  7 Ce atoms: 216 structures
  8 Ce atoms: 89 structures
  9 Ce atoms: 28 structures
